# Logistic Regression Feature Importance

Цель notebook: посмотреть, на какие признаки сильнее всего опирается текущая логистическая регрессия для прогноза `target_had_accident_next_month`.

Используется тот же набор, что в последнем эксперименте: `road_length_total_m + road_share_* + road/junction/speed/calendar`, но без per-type `road_length_*_m`.

## Setup

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path.cwd().parents[0]
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from road_accident_risk.osm_road_features import MOTOR_ROAD_CLASSES

pd.set_option("display.max_rows", 120)
pd.set_option("display.max_columns", 120)
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

## Load Data And Feature Set

In [ ]:
PANEL_PATH = PROJECT_ROOT / "parquet_output" / "h3_month_road_features.parquet"
TARGET = "target_had_accident_next_month"
TRAIN_END = "2023-01-01"

panel = pd.read_parquet(PANEL_PATH)
panel["month"] = pd.to_datetime(panel["month"])
panel[TARGET] = panel[TARGET].astype(int)

per_type_road_lengths = {f"road_length_{highway}_m" for highway in MOTOR_ROAD_CLASSES}
excluded = {"h3", "month", TARGET} | per_type_road_lengths
features = [column for column in panel.columns if column not in excluded]

print("panel shape:", panel.shape)
print("feature count:", len(features))
print("excluded per-type length columns:", len(per_type_road_lengths & set(panel.columns)))

train = panel[panel["month"] < TRAIN_END].copy()
test = panel[panel["month"] >= TRAIN_END].copy()

X_train = train[features]
y_train = train[TARGET]
X_test = test[features]
y_test = test[TARGET]

print("train:", train.shape, "positive rate:", y_train.mean())
print("test:", test.shape, "positive rate:", y_test.mean())

In [ ]:
pd.DataFrame({"feature": features})

## Train Logistic Regression

Коэффициенты ниже интерпретируются после `StandardScaler`, то есть это вклад изменения признака на одно стандартное отклонение. Так сравнивать признаки корректнее, чем сырые коэффициенты в разных единицах измерения.

In [ ]:
model = Pipeline([
    ("scaler", StandardScaler()),
    (
        "model",
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            solver="lbfgs",
            n_jobs=None,
        ),
    ),
])

model.fit(X_train, y_train)
test_scores = model.predict_proba(X_test)[:, 1]

metrics = {
    "roc_auc": roc_auc_score(y_test, test_scores),
    "pr_auc": average_precision_score(y_test, test_scores),
}
metrics

## Coefficients

In [ ]:
coef = model.named_steps["model"].coef_[0]
coef_df = pd.DataFrame({
    "feature": features,
    "coef": coef,
})
coef_df["abs_coef"] = coef_df["coef"].abs()
coef_df["direction"] = np.where(coef_df["coef"] >= 0, "risk_up", "risk_down")
coef_df = coef_df.sort_values("abs_coef", ascending=False).reset_index(drop=True)
coef_df.head(40)

In [ ]:
def plot_barh(df: pd.DataFrame, title: str, value_col: str = "coef") -> None:
    plot_df = df.iloc[::-1]
    colors = np.where(plot_df[value_col] >= 0, "#b84a3a", "#2f7f8f")
    fig, ax = plt.subplots(figsize=(11, max(5, len(plot_df) * 0.32)))
    ax.barh(plot_df["feature"], plot_df[value_col], color=colors)
    ax.axvline(0, color="#1f2933", linewidth=1)
    ax.set_title(title)
    ax.set_xlabel(value_col)
    plt.tight_layout()

top_coef = coef_df.head(30).sort_values("coef")
plot_barh(top_coef, "Top 30 standardized coefficients")

In [ ]:
top_positive = coef_df.sort_values("coef", ascending=False).head(25)
top_negative = coef_df.sort_values("coef", ascending=True).head(25)

display(top_positive[["feature", "coef", "abs_coef"]])
display(top_negative[["feature", "coef", "abs_coef"]])

## Feature Groups

Группировка ниже не является статистическим тестом. Это быстрый способ понять, какие семейства признаков дают самые крупные коэффициенты в линейной модели.

In [ ]:
def feature_group(name: str) -> str:
    if name in {"year", "month_num"}:
        return "calendar"
    if name.startswith("road_share_"):
        return "road_type_share"
    if name.startswith("maxspeed"):
        return "maxspeed"
    if name.startswith("lanes") or name == "lane_meters_estimate":
        return "lanes_capacity"
    if "junction" in name or "class_delta" in name:
        return "junction_graph"
    if name.endswith("_share"):
        return "road_attribute_share"
    if name.endswith("_count"):
        return "point_count"
    if name.endswith("_length_m") or name == "road_length_total_m":
        return "road_length"
    return "other"

coef_df["group"] = coef_df["feature"].map(feature_group)
group_df = (
    coef_df.groupby("group")
    .agg(
        feature_count=("feature", "size"),
        abs_coef_sum=("abs_coef", "sum"),
        abs_coef_mean=("abs_coef", "mean"),
        max_abs_coef=("abs_coef", "max"),
    )
    .sort_values("abs_coef_sum", ascending=False)
)
group_df

In [ ]:
plot_df = group_df.reset_index().sort_values("abs_coef_sum")
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(plot_df["group"], plot_df["abs_coef_sum"], color="#3c6e71")
ax.set_title("Coefficient mass by feature group")
ax.set_xlabel("sum(abs(standardized coefficient))")
plt.tight_layout()

## Permutation Importance

Permutation importance медленнее, но ближе к вопросу “насколько падает качество, если сломать один признак”. Для скорости используется сэмпл test rows. Увеличьте `SAMPLE_SIZE`, если нужна более стабильная оценка.

In [ ]:
SAMPLE_SIZE = 80_000
RANDOM_STATE = 42

if len(X_test) > SAMPLE_SIZE:
    sample_idx = X_test.sample(SAMPLE_SIZE, random_state=RANDOM_STATE).index
    X_perm = X_test.loc[sample_idx]
    y_perm = y_test.loc[sample_idx]
else:
    X_perm = X_test
    y_perm = y_test

print("permutation sample:", X_perm.shape)
print("positive rate:", y_perm.mean())

In [ ]:
perm = permutation_importance(
    model,
    X_perm,
    y_perm,
    scoring="average_precision",
    n_repeats=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

perm_df = pd.DataFrame({
    "feature": features,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
})
perm_df["group"] = perm_df["feature"].map(feature_group)
perm_df = perm_df.sort_values("importance_mean", ascending=False).reset_index(drop=True)
perm_df.head(40)

In [ ]:
top_perm = perm_df.head(30).iloc[::-1]
fig, ax = plt.subplots(figsize=(11, max(5, len(top_perm) * 0.32)))
ax.barh(top_perm["feature"], top_perm["importance_mean"], xerr=top_perm["importance_std"], color="#6f8f72")
ax.axvline(0, color="#1f2933", linewidth=1)
ax.set_title("Top permutation importance, scoring=average_precision")
ax.set_xlabel("PR-AUC decrease after permutation")
plt.tight_layout()

In [ ]:
perm_group_df = (
    perm_df.groupby("group")
    .agg(
        feature_count=("feature", "size"),
        importance_sum=("importance_mean", "sum"),
        importance_mean=("importance_mean", "mean"),
        max_importance=("importance_mean", "max"),
    )
    .sort_values("importance_sum", ascending=False)
)
perm_group_df

## Export Tables

Следующая ячейка сохраняет таблицы важности в `reports/`, чтобы их можно было сравнивать между запусками.

In [ ]:
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(exist_ok=True)

coef_path = REPORTS_DIR / "logreg_feature_coefficients_without_type_lengths.csv"
perm_path = REPORTS_DIR / "logreg_permutation_importance_without_type_lengths.csv"

coef_df.to_csv(coef_path, index=False)
perm_df.to_csv(perm_path, index=False)

print(coef_path)
print(perm_path)